In [1]:
import numpy as np
import random

class TicTacToe:
    def __init__(self):
        self.board = np.zeros((3, 3), dtype=int) # 0: empty, 1: X (Agent), -1: O (Opponent)
        self.winner = None
        self.game_over = False

    def reset(self):
        self.board = np.zeros((3, 3), dtype=int)
        self.winner = None
        self.game_over = False
        return self.get_state()

    def get_state(self):
        return str(self.board.flatten())

    def available_actions(self):
        return [(r, c) for r in range(3) for c in range(3) if self.board[r, c] == 0]

    def make_move(self, action, player):
        r, c = action
        if self.board[r, c] == 0:
            self.board[r, c] = player
            self.check_game_over()
            return True
        return False

    def check_game_over(self):
        for i in range(3):
            if abs(sum(self.board[i, :])) == 3:
                self.winner = self.board[i, 0]
                self.game_over = True
                return
            if abs(sum(self.board[:, i])) == 3:
                self.winner = self.board[0, i]
                self.game_over = True
                return
        if abs(self.board[0,0] + self.board[1,1] + self.board[2,2]) == 3:
            self.winner = self.board[1,1]
            self.game_over = True
            return
        if abs(self.board[0,2] + self.board[1,1] + self.board[2,0]) == 3:
            self.winner = self.board[1,1]
            self.game_over = True
            return
        if len(self.available_actions()) == 0:
            self.winner = 0 # Tie
            self.game_over = True

class QLearningAgent:
    def __init__(self, alpha=0.1, gamma=0.9, epsilon=1.0):
        self.q_table = {}
        self.alpha = alpha     # Learning rate
        self.gamma = gamma     # Discount factor
        self.epsilon = epsilon # Exploration rate

    def get_q(self, state, action):
        return self.q_table.get((state, action), 0.0)

    def choose_action(self, state, available_actions):
        if random.uniform(0, 1) < self.epsilon:
            return random.choice(available_actions) # Explore
        q_values = [self.get_q(state, a) for a in available_actions]
        max_q = max(q_values)
        # Handle ties by picking randomly among best actions
        best_actions = [available_actions[i] for i, q in enumerate(q_values) if q == max_q]
        return random.choice(best_actions) # Exploit

    def update_q(self, state, action, reward, next_state, next_available):
        current_q = self.get_q(state, action)
        max_next_q = max([self.get_q(next_state, a) for a in next_available]) if next_available else 0.0
        new_q = current_q + self.alpha * (reward + self.gamma * max_next_q - current_q)
        self.q_table[(state, action)] = new_q

# --- Training the Agent ---
env = TicTacToe()
agent = QLearningAgent()

episodes = 50000
for episode in range(episodes):
    state = env.reset()
    while not env.game_over:
        actions = env.available_actions()
        action = agent.choose_action(state, actions)

        # Agent's move (Player 1)
        env.make_move(action, 1)
        next_state = env.get_state()

        if env.game_over:
            if env.winner == 1:
                agent.update_q(state, action, 1.0, next_state, [])
            elif env.winner == 0:
                agent.update_q(state, action, 0.5, next_state, [])
            break

        # Opponent's random move (Player -1)
        opponent_actions = env.available_actions()
        if opponent_actions:
            opp_action = random.choice(opponent_actions)
            env.make_move(opp_action, -1)
            next_state = env.get_state()

            if env.game_over:
                if env.winner == -1:
                    agent.update_q(state, action, -1.0, next_state, [])
                    break
                elif env.winner == 0:
                    agent.update_q(state, action, 0.5, next_state, [])
                    break

        next_actions = env.available_actions()
        agent.update_q(state, action, 0.0, next_state, next_actions)
        state = next_state

    # Decay epsilon
    if agent.epsilon > 0.01:
        agent.epsilon *= 0.9992

print("Training Complete! The RL Agent is ready to play.")

Training Complete! The RL Agent is ready to play.


In [2]:
#Test and Play Against Your AI
def play_against_ai(agent):
    env = TicTacToe()
    state = env.reset()
    print("Game Started! You are 'O' (-1) and AI is 'X' (1).")
    print("Positions are indexed from (0,0) to (2,2) top-left to bottom-right.\n")

    while not env.game_over:
        print(env.board)

        # AI Turn
        actions = env.available_actions()
        ai_action = agent.choose_action(state, actions)
        env.make_move(ai_action, 1)
        state = env.get_state()

        if env.game_over:
            break

        print("\nAI has made its move:")
        print(env.board)

        # Human Turn
        human_actions = env.available_actions()
        valid = False
        while not valid:
            try:
                r = int(input("Enter row (0, 1, 2): "))
                c = int(input("Enter col (0, 1, 2): "))
                if (r, c) in human_actions:
                    env.make_move((r, c), -1)
                    state = env.get_state()
                    valid = True
                else:
                    print("Invalid move! Spot taken or out of bounds. Try again.")
            except ValueError:
                print("Please enter valid numbers.")

    print("\n--- Game Over ---")
    print(env.board)
    if env.winner == 1:
        print("AI Wins!")
    elif env.winner == -1:
        print("You Win!")
    else:
        print("It's a Tie!")

# Run this to play!
play_against_ai(agent)

Game Started! You are 'O' (-1) and AI is 'X' (1).
Positions are indexed from (0,0) to (2,2) top-left to bottom-right.

[[0 0 0]
 [0 0 0]
 [0 0 0]]

AI has made its move:
[[0 0 1]
 [0 0 0]
 [0 0 0]]
Enter row (0, 1, 2): 0
Enter col (0, 1, 2): 1
[[ 0 -1  1]
 [ 0  0  0]
 [ 0  0  0]]

AI has made its move:
[[ 0 -1  1]
 [ 0  0  0]
 [ 0  0  1]]
Enter row (0, 1, 2): 1
Enter col (0, 1, 2): 2
[[ 0 -1  1]
 [ 0  0 -1]
 [ 0  0  1]]

AI has made its move:
[[ 0 -1  1]
 [ 0  1 -1]
 [ 0  0  1]]
Enter row (0, 1, 2): 2
Enter col (0, 1, 2): 0
[[ 0 -1  1]
 [ 0  1 -1]
 [-1  0  1]]

--- Game Over ---
[[ 1 -1  1]
 [ 0  1 -1]
 [-1  0  1]]
AI Wins!
